In [ ]:
pip install openai pandas scikit-learn


In [ ]:
import pandas as pd
import time
from openai import OpenAI
import os
os.environ["OPENAI_API_KEY"] = "" #Enter your API Key here

Data Processing

In [ ]:
#Step 1 : Loading Documents and Test logs
TestLogs = pd.read_csv("Test_dataset.csv")
TestLogs_Cols = TestLogs[['Content', 'EventTemplate','Source','Category']] # Select category for Seen datasets only
ground_truth = TestLogs['EventTemplate'].tolist()




Inference Zero-Shot


In [ ]:
"""Inference"""
client = OpenAI()

# ========== Prompt Format ==========
prompt_template = """You are a log parsing assistant. Your task is to extract the template of the given log message "
            "by replacing dynamic parts like timestamps, IDs, IP addresses, or numeric values with the '<*>' placeholder. "
            "If the log message is fully static and has no dynamic parts, return it as-is with no placeholders. "
            "Return ONLY the extracted template with no explanations, reasoning, or additional text."

### Log Message to Parse:
```
{log}
```

### Response:
"""

def extract_log_templates_gpt41(new_logs):
    results = []
    for idx, row in enumerate(new_logs.itertuples(), start=1):
        log = row.Content
        category = row.Category
        source = row.Source

        formatted_prompt = prompt_template.format(log=log)

        print(f"\nProcessing Log {idx} ({source})...")
        print("=" * 80)
        print(f"**Log Message:** {log}")

        # Query GPT-4.1
        start_time = time.time()
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": formatted_prompt}],
            max_tokens=300,
            temperature=0
        )
        elapsed_time = time.time() - start_time

        llm_response = response.choices[0].message.content.strip()
        extracted_template = llm_response.replace("`", "").strip()

        print(f"**LLM Response:** {llm_response}")
        print(f"**Extracted Template:** {extracted_template}")
        print(f"Inference Time: {elapsed_time:.2f} seconds")
        print("=" * 80)

        results.append({
            "Log": log,
            "Extracted Template": extracted_template,
            "Category": category,
            "Source": source
        })

    return results

Inference Few-Shot

In [ ]:
"""Inference"""
client = OpenAI()
# ========== Prompt Format ==========
prompt_template = """You are a log parsing assistant. Your task is to extract the template of the given log message "
            "by replacing dynamic parts like timestamps, IDs, IP addresses, or numeric values with the '<*>' placeholder. "
            "Use the examples in the context to guide your output. "
            "If the log message is fully static and has no dynamic parts, return it as-is with no placeholders. "
            "Return ONLY the extracted template with no explanations, reasoning, or additional text."

### Context:
Example: NIFF: node node-134 has detected an available network connection on network 8.127.0.0 via interface ee0
Template: NIFF: node node-<*> has detected an available network connection on network <*> via interface ee0

Example: connection from 192.168.1.100 () at Mon Aug 8 09:15:43 2006
Template: connection from <*> (<*>) at <*> <*> <*> <*>:<*>:<*> <*>


### Log Message to Parse:
```
{log}
```

### Response:
"""

def extract_log_templates_gpt41(new_logs):
    results = []
    for idx, row in enumerate(new_logs.itertuples(), start=1):
        log = row.Content
        category = row.Category
        source = row.Source

        formatted_prompt = prompt_template.format(log=log)

        print(f"\nProcessing Log {idx} ({source})...")
        print("=" * 80)
        print(f"**Log Message:** {log}")
        print(f"**LLM Input Prompt:**\n{formatted_prompt}\n")


        # Query GPT-4.1
        start_time = time.time()
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": formatted_prompt}],
            max_tokens=300,
            temperature=0
        )
        elapsed_time = time.time() - start_time

        llm_response = response.choices[0].message.content.strip()
        extracted_template = llm_response.replace("`", "").strip()

        print(f"**LLM Response:** {llm_response}")
        print(f"**Extracted Template:** {extracted_template}")
        print(f"Inference Time: {elapsed_time:.2f} seconds")
        print("=" * 80)

        results.append({
            "Log": log,
            "Extracted Template": extracted_template,
            "Category": category,
            "Source": source
        })

    return results

In [ ]:
# Run the pipeline
start_time = time.time()  # Start timer
results = extract_log_templates_gpt41(TestLogs_Cols)
end_time = time.time()  # End timer
total_inference_time = end_time - start_time
print(f"Total inference time: {total_inference_time:.2f} seconds")

# Save predictions
pred_df = pd.DataFrame(results)
pred_df.to_csv("GPT4.1_logparsing_results.csv", index=False)
print("✅ Extracted templates saved successfully (GPT-4.1).")